# Temporal GNNs for Misinformation Detection — TAGFN Pipeline
**Emmanuel Asiedu Yeboah · 22424640 · MSc Data Science, University of Ghana**

Consolidated, ordered pipeline. After any Colab disconnect, use
**Runtime → Run all** to rebuild the whole validated state in one click.

Order: installs → load & group TAGFN → statistics → construction (windows +
four model representations) → leakage test (the masking proof) → models (next).

Set `SUBSET` in the first code cell: `politifact` (primary), `gossipcop`
(scale-up), or `fakeddit` (scalability tier — large, run selectively).


## 0. Installs

In [ ]:
!pip install -q datasets huggingface_hub torch_geometric
print("installs done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.2 MB/s eta 0:00:00
installs done


## 1. Load and group TAGFN
Session 1
Downloads only the small structural arrays (not the big text parquet) and
reconstructs per-cascade graphs. Verifies the tree structure via the
edge-count identity (nodes − edges = #graphs).

In [ ]:
from huggingface_hub import hf_hub_download
from collections import deque, defaultdict
import numpy as np, random

REPO   = "kayzliu/TAGFN"
SUBSET = "politifact"        # <-- change to "gossipcop" / "fakeddit" as needed

def load_subset(sub):
    ngi = np.load(hf_hub_download(REPO, f"{sub}/node_graph_id.npy", repo_type="dataset"))
    lab = np.load(hf_hub_download(REPO, f"{sub}/graph_labels.npy",  repo_type="dataset"))
    tim = np.load(hf_hub_download(REPO, f"{sub}/node_time.npy",     repo_type="dataset"))
    a_path = hf_hub_download(REPO, f"{sub}/A.txt", repo_type="dataset")
    edges = []
    with open(a_path) as fh:
        for line in fh:
            line = line.strip()
            if not line: continue
            if "," in line: x, y = line.split(",")[:2]
            else:           p = line.split(); x, y = p[0], p[1]
            edges.append((int(x), int(y)))
    return ngi, lab.astype(int).ravel(), tim, np.array(edges, dtype=np.int64)

node_graph_id, graph_labels, node_time, edges = load_subset(SUBSET)
G = graph_labels.shape[0]

# group nodes and edges by graph
order = np.argsort(node_graph_id, kind="stable")
sorted_gids = node_graph_id[order]
bnd = np.searchsorted(sorted_gids, np.arange(G + 1))
nodes_of = {g: order[bnd[g]:bnd[g+1]] for g in range(G)}
edge_gid = node_graph_id[edges[:, 0]]
edges_of = defaultdict(list)
for (u, v), g in zip(edges, edge_gid):
    edges_of[g].append((int(u), int(v)))

n_nodes, n_edges = node_graph_id.shape[0], edges.shape[0]
print(f"{SUBSET}: {G:,} graphs, {n_nodes:,} nodes, {n_edges:,} edges")
print("tree check (nodes - edges == #graphs):", n_nodes - n_edges == G)

politifact/node_graph_id.npy: reconstructing file:   0%|          |  0.00B /  329kB            

politifact/node_graph_id.npy: downloading bytes:           |  0.00B            

politifact/graph_labels.npy: reconstructing file:   0%|          |  0.00B / 2.64kB            

politifact/graph_labels.npy: downloading bytes:           |  0.00B            

politifact/node_time.npy: reconstructing file:   0%|          |  0.00B /  329kB            

politifact/node_time.npy: downloading bytes:           |  0.00B            

A.txt:   0%|          | 0.00/507k [00:00<?, ?B/s]

politifact: 314 graphs, 41,054 nodes, 40,740 edges
tree check (nodes - edges == #graphs): True


## 2. Statistics (quick sanity)
Class balance and cascade-size distribution. Confirms the numbers used in
Table 3.1 and the small-cascade tail that motivates the observation-window
floor.

In [ ]:
sizes = np.array([len(nodes_of[g]) for g in range(G)])
pct_fake = 100.0 * graph_labels.sum() / G
print(f"% fake: {pct_fake:.1f}%")
print(f"cascade size -> min {sizes.min()}, median {int(np.median(sizes))}, "
      f"mean {sizes.mean():.1f}, max {sizes.max()}")
for thr in (5, 10, 20, 50):
    print(f"  % cascades < {thr:>2} nodes: {100*(sizes<thr).mean():5.1f}%")

% fake: 50.0%
cascade size -> min 3, median 78, mean 130.7, max 497
  % cascades <  5 nodes:   3.5%
  % cascades < 10 nodes:   8.9%
  % cascades < 20 nodes:  18.5%
  % cascades < 50 nodes:  39.8%


## 3. Construction: windows + four model representations
Turns each cascade into the four model inputs under masking. The observation
window is `max(k%, FLOOR)` events, taken in chronological order. Node features
are structural + temporal only (no text, by design), with time re-anchored to
the first observed engagement (the news root carries a placeholder timestamp).

In [ ]:
FRACTIONS = [0.10, 0.20, 0.30, 0.50]   # early-detection windows
FLOOR     = 5                           # minimum observed events

def cascade_event_stream(g):
    root = int(nodes_of[g].min())
    tmap = {int(n): int(node_time[n]) for n in nodes_of[g]}
    evs = []
    for (u, v) in edges_of[g]:
        if tmap.get(u, 0) >= tmap.get(v, 0): child, par = u, v
        else:                                child, par = v, u
        evs.append((int(child), int(par), tmap.get(int(child), 0)))
    evs.sort(key=lambda e: (e[2], e[0]))
    return evs, root, tmap

def observed_prefix(stream, k, floor=FLOOR):
    n = len(stream)
    return stream[:min(max(int(np.floor(k * n)), floor), n)]

def features_from_prefix(prefix, root, tmap):
    nodes = {root}; adj = defaultdict(list)
    for child, par, t in prefix:
        nodes.add(child); nodes.add(par)
        adj[child].append(par); adj[par].append(child)
    nodes = sorted(nodes)
    degree = {n: len(adj[n]) for n in nodes}
    parent = {root: None}; depth = {root: 0}; bfs = []
    q = deque([root]); seen = {root}
    while q:
        u = q.popleft(); bfs.append(u)
        for w in adj[u]:
            if w not in seen:
                seen.add(w); parent[w] = u; depth[w] = depth[u] + 1; q.append(w)
    subtree = {n: 1 for n in nodes}
    for u in reversed(bfs):
        if parent.get(u) is not None: subtree[parent[u]] += subtree[u]
    non_root = [tmap[n] for n in nodes if n != root and n in tmap]
    t_first = min(non_root) if non_root else 0
    rt = lambda n: 0 if n == root else tmap.get(n, 0) - t_first
    feats = {}
    for n in nodes:
        p = parent.get(n)
        feats[n] = [depth.get(n, -1), degree[n], rt(n),
                    0 if p is None else rt(n) - rt(p), subtree[n]]
    return feats, nodes

def build_representations(g, k):
    stream, root, tmap = cascade_event_stream(g)
    prefix = observed_prefix(stream, k)
    feats, nodes = features_from_prefix(prefix, root, tmap)
    temporal = [(c, p, t, tuple(feats[c])) for (c, p, t) in prefix]
    static = {"nodes": tuple(nodes),
              "edges": frozenset((c, p) for (c, p, t) in prefix),
              "feats": tuple(tuple(feats[n]) for n in nodes)}
    sequence = [tuple(feats[c]) for (c, p, t) in prefix]
    arr = np.array([feats[n] for n in nodes], dtype=float)
    flat = tuple(np.concatenate([arr.mean(0), arr.max(0), arr.min(0),
                                 [len(nodes)]]).tolist())
    return {"temporal": temporal, "static": static, "sequence": sequence,
            "flat": flat, "label": int(graph_labels[g])}

# sanity: shapes for graph 0
for k in FRACTIONS:
    r = build_representations(0, k)
    print(f"k={int(k*100):>2}%: temporal={len(r['temporal'])}, "
          f"static={len(r['static']['nodes'])}n/"
          f"{len(r['static']['edges'])}e, seq={len(r['sequence'])}, "
          f"flat={len(r['flat'])}d, label={r['label']}")

k=10%: temporal=49, static=50n/49e, seq=49, flat=16d, label=0
k=20%: temporal=99, static=100n/99e, seq=99, flat=16d, label=0
k=30%: temporal=148, static=149n/148e, seq=148, flat=16d, label=0
k=50%: temporal=248, static=249n/248e, seq=248, flat=16d, label=0


## 4. Leakage test — the masking proof (defence question D1)
Perturbs every event after the observation cut-off (reorder, push times far
forward, delete some) and confirms all four representations are unchanged.
A pass means nothing after the cut-off influences any prediction.

In [ ]:
def leakage_test_real(g, k, seed=0):
    rng = np.random.default_rng(seed)
    stream, root, tmap = cascade_event_stream(g)
    prefix = observed_prefix(stream, k); tcut = prefix[-1][2]
    before = build_representations(g, k)
    future = list(stream[len(prefix):]); rng.shuffle(future)
    future = [(c, p, t + int(rng.integers(10**6, 10**7))) for (c, p, t) in future]
    if len(future) > 1:
        d = int(rng.integers(0, len(future))); future = future[:d] + future[d+1:]
    merged = sorted(prefix + future, key=lambda e: (e[2], e[0]))
    prefix2 = [e for e in merged if e[2] <= tcut]
    f2, n2 = features_from_prefix(prefix2, root, tmap)
    t2 = [(c, p, t, tuple(f2[c])) for (c, p, t) in prefix2]
    s2 = {"nodes": tuple(n2), "edges": frozenset((c, p) for (c, p, t) in prefix2),
          "feats": tuple(tuple(f2[n]) for n in n2)}
    q2 = [tuple(f2[c]) for (c, p, t) in prefix2]
    a2 = np.array([f2[n] for n in n2], dtype=float)
    fl2 = tuple(np.concatenate([a2.mean(0), a2.max(0), a2.min(0), [len(n2)]]).tolist())
    return (before["temporal"] == t2 and before["static"] == s2
            and before["sequence"] == q2 and before["flat"] == fl2)

random.seed(0)
sample = random.sample(range(G), min(50, G))
ok_all = True
for k in FRACTIONS:
    ok = all(leakage_test_real(g, k, s) for g in sample for s in range(3))
    ok_all &= ok
    print(f"k={int(k*100):>2}%: {'PASS' if ok else 'FAIL'}")
print("RESULT:", "NO LEAKAGE across all cascades and windows." if ok_all
      else "LEAKAGE — investigate before proceeding.")

k=10%: PASS
k=20%: PASS
k=30%: PASS
k=50%: PASS
RESULT: NO LEAKAGE across all cascades and windows.


## 5. Models — *next session*
Section 3.5 will add: masking-aware normalisation, the temporal GNN (TGAT/TGN),
the four baselines (logistic regression, LSTM, static GCN, flat), and the first
training run on PolitiFact. Nothing above needs to change — the four
representations produced in Section 3 are the model inputs.

In [ ]:
# ===== SECTION 5.0 — Setup: reproducibility, GPU, Drive, checkpointing =====
import os, random, numpy as np, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False   # reproducible, at a small speed cost

# GPU — record which one (matters for the scalability chapter)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device,
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# verify PyTorch Geometric is importable (library check)
import torch_geometric
print("PyTorch Geometric:", torch_geometric.__version__)

# mount Google Drive so models/results survive disconnects
from google.colab import drive
drive.mount("/content/drive")

# results + checkpoint folder (edit the path to a folder you like)
SAVE_DIR = "/content/drive/MyDrive/thesis_tagfn"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Saving results to:", SAVE_DIR)
print(f"Reproducibility: fixed seed = {SEED}")

Device: cuda | Tesla T4
PyTorch Geometric: 2.8.0.post1
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving results to: /content/drive/MyDrive/thesis_tagfn
Reproducibility: fixed seed = 42


SECTION 5.1 — Masking-aware feature normalisation

In [ ]:
TIME_COLS = [2, 3]   # the two time features span 0..millions -> compress first

def normalise_observed(feat_rows):
    """feat_rows: list/array of node feature vectors from ONE observed window.
    Returns the normalised matrix. Stats derive only from these rows."""
    X = np.asarray(feat_rows, dtype=float).copy()
    for c in TIME_COLS:
        X[:, c] = np.log1p(np.clip(X[:, c], 0, None))   # compress huge seconds
    mu = X.mean(0); sd = X.std(0); sd[sd == 0] = 1.0     # guard zero variance
    return (X - mu) / sd

# quick check on a real PolitiFact window
_r = build_representations(0, 0.30)
_feats = [list(f) for f in _r["static"]["feats"]]
_Xn = normalise_observed(_feats)
print("graph 0 @30%: feature matrix", _Xn.shape,
      "| per-col mean≈0:", np.round(_Xn.mean(0), 3),
      "| per-col sd≈1:", np.round(_Xn.std(0), 3))

graph 0 @30%: feature matrix (149, 5) | per-col mean≈0: [ 0. -0.  0. -0.  0.] | per-col sd≈1: [1. 1. 1. 1. 1.]


SECTION 5.2 — Training harness + logistic-regression baseline

In [ ]:
# Builds the flat-vector dataset for one observation window, trains logistic
# regression, evaluates on the OFFICIAL test split. This validates the whole
# train/evaluate/metrics loop with the simplest model before we add TGAT.
from huggingface_hub import hf_hub_download
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# --- load the official train/val/test split indices for this subset ---
def load_split(sub):
    tr = np.load(hf_hub_download(REPO, f"{sub}/train_idx.npy", repo_type="dataset"))
    va = np.load(hf_hub_download(REPO, f"{sub}/val_idx.npy",   repo_type="dataset"))
    te = np.load(hf_hub_download(REPO, f"{sub}/test_idx.npy",  repo_type="dataset"))
    return tr.astype(int), va.astype(int), te.astype(int)

train_idx, val_idx, test_idx = load_split(SUBSET)
print(f"splits -> train {len(train_idx)}, val {len(val_idx)}, test {len(test_idx)}")

# --- build flat-vector dataset at a given window k ---
def flat_dataset(graph_ids, k):
    X, y = [], []
    for g in graph_ids:
        r = build_representations(int(g), k)
        X.append(r["flat"]); y.append(r["label"])
    return np.array(X, dtype=float), np.array(y, dtype=int)

def evaluate(model, X, y):
    p = model.predict_proba(X)[:, 1]
    yhat = (p >= 0.5).astype(int)
    return {"accuracy": accuracy_score(y, yhat),
            "macro_f1": f1_score(y, yhat, average="macro"),
            "auc": roc_auc_score(y, p)}

# --- run logistic regression across all four windows ---
print("\nLogistic regression (flat features) on", SUBSET)
print(f"{'window':>7}{'accuracy':>10}{'macro_f1':>10}{'auc':>8}")
lr_results = {}
for k in FRACTIONS:
    Xtr, ytr = flat_dataset(train_idx, k)
    Xte, yte = flat_dataset(test_idx, k)
    clf = LogisticRegression(max_iter=1000)
    clf.fit(Xtr, ytr)
    m = evaluate(clf, Xte, yte)
    lr_results[k] = m
    print(f"{int(k*100):>6}%{m['accuracy']:>10.3f}{m['macro_f1']:>10.3f}{m['auc']:>8.3f}")

politifact/train_idx.npy: reconstructing file:   0%|          |  0.00B /   624B            

politifact/train_idx.npy: downloading bytes:           |  0.00B            

politifact/val_idx.npy: reconstructing file:   0%|          |  0.00B /   376B            

politifact/val_idx.npy: downloading bytes:           |  0.00B            

politifact/test_idx.npy: reconstructing file:   0%|          |  0.00B / 1.90kB            

politifact/test_idx.npy: downloading bytes:           |  0.00B            

splits -> train 62, val 31, test 221

Logistic regression (flat features) on politifact
 window  accuracy  macro_f1     auc
    10%     0.525     0.445   0.634
    20%     0.538     0.454   0.716
    30%     0.624     0.618   0.722
    50%     0.661     0.657   0.738


SECTION 5.3 — TGAT (main temporal model)

In [ ]:

import math, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

class TimeEncoding(nn.Module):
    def __init__(self, dim):
        super().__init__(); self.w = nn.Linear(1, dim)
    def forward(self, t): return torch.cos(self.w(t))

class TemporalAttentionLayer(nn.Module):
    def __init__(self, in_dim, time_dim, out_dim, heads=2, dropout=0.3):
        super().__init__()
        self.heads, self.out_dim = heads, out_dim
        self.q = nn.Linear(in_dim, out_dim*heads)
        self.k = nn.Linear(in_dim+time_dim, out_dim*heads)
        self.v = nn.Linear(in_dim+time_dim, out_dim*heads)
        self.time_enc = TimeEncoding(time_dim)
        self.out = nn.Linear(out_dim*heads, out_dim); self.drop = nn.Dropout(dropout)
    def forward(self, x, edge_index, edge_time, num_nodes):
        src, dst = edge_index[0], edge_index[1]
        te = self.time_enc(edge_time)
        q = self.q(x)[dst]; kv = torch.cat([x[src], te], -1)
        k = self.k(kv); v = self.v(kv)
        H, O = self.heads, self.out_dim
        q, k, v = q.view(-1,H,O), k.view(-1,H,O), v.view(-1,H,O)
        s = (q*k).sum(-1)/math.sqrt(O); s = s - s.max()
        e = torch.exp(s)
        denom = torch.zeros(num_nodes, H, device=x.device).index_add_(0, dst, e)+1e-9
        alpha = e/denom[dst]
        agg = torch.zeros(num_nodes, H, O, device=x.device).index_add_(
            0, dst, alpha.unsqueeze(-1)*v).reshape(num_nodes, H*O)
        return self.drop(F.relu(self.out(agg)))

class TGAT(nn.Module):
    def __init__(self, in_dim, hid=32, time_dim=16, layers=2, heads=2, dropout=0.3):
        super().__init__()
        self.inp = nn.Linear(in_dim, hid)
        self.layers = nn.ModuleList([TemporalAttentionLayer(hid, time_dim, hid, heads, dropout)
                                     for _ in range(layers)])
        self.head = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hid, 1))
    def forward(self, x, edge_index, edge_time, batch, num_graphs):
        h = F.relu(self.inp(x))
        for layer in self.layers:
            h = h + layer(h, edge_index, edge_time, x.size(0))
        pooled = torch.zeros(num_graphs, h.size(1), device=x.device)
        counts = torch.zeros(num_graphs, 1, device=x.device)
        pooled.index_add_(0, batch, h)
        counts.index_add_(0, batch, torch.ones(batch.size(0),1, device=x.device))
        return self.head(pooled/(counts+1e-9)).squeeze(-1)

# --- turn our temporal representation into TGAT tensors, with masking-aware norm ---
def graphs_to_tensors(graph_ids, k):
    xs, srcs, dsts, ets, batch, ys = [], [], [], [], [], []
    off = 0; gi = 0
    for g in graph_ids:
        r = build_representations(int(g), k)
        temp = r["temporal"]                 # list of (child, par, time, feat)
        # local node ids for this graph
        node_list = list(r["static"]["nodes"])
        idx = {n:i for i,n in enumerate(node_list)}
        feats = normalise_observed([list(f) for f in r["static"]["feats"]])
        n = len(node_list)
        xs.append(feats)
        for (c,p,t,fe) in temp:
            srcs.append(idx[c]+off); dsts.append(idx[p]+off); ets.append(t)
        batch += [gi]*n; ys.append(r["label"]); off += n; gi += 1
    X  = torch.tensor(np.concatenate(xs), dtype=torch.float, device=device)
    ei = torch.tensor(np.array([srcs, dsts]), dtype=torch.long, device=device)
    # log1p-compress edge times (same rationale as node time features)
    et = torch.tensor(np.log1p(np.array(ets, dtype=float)), dtype=torch.float,
                      device=device).unsqueeze(1)
    b  = torch.tensor(np.array(batch), dtype=torch.long, device=device)
    y  = torch.tensor(np.array(ys), dtype=torch.float, device=device)
    return X, ei, et, b, y, gi

def train_tgat(k, epochs=200, patience=20, seed=SEED):
    torch.manual_seed(seed)
    Xtr,eitr,ettr,btr,ytr,ntr = graphs_to_tensors(train_idx, k)
    Xva,eiva,etva,bva,yva,nva = graphs_to_tensors(val_idx, k)
    Xte,eite,ette,bte,yte,nte = graphs_to_tensors(test_idx, k)
    model = TGAT(in_dim=5).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-2, weight_decay=1e-4)
    best_auc, best_state, wait = -1, None, 0
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xtr,eitr,ettr,btr,ntr), ytr)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            va = torch.sigmoid(model(Xva,eiva,etva,bva,nva)).cpu().numpy()
        try: vauc = roc_auc_score(yva.cpu().numpy(), va)
        except: vauc = 0.5
        if vauc > best_auc: best_auc, best_state, wait = vauc, {kk:v.clone() for kk,v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        p = torch.sigmoid(model(Xte,eite,ette,bte,nte)).cpu().numpy()
    yhat = (p>=0.5).astype(int); yt = yte.cpu().numpy().astype(int)
    return {"accuracy":accuracy_score(yt,yhat),
            "macro_f1":f1_score(yt,yhat,average="macro"),
            "auc":roc_auc_score(yt,p)}

print("TGAT on", SUBSET, "(early-stopped on val)")
print(f"{'window':>7}{'accuracy':>10}{'macro_f1':>10}{'auc':>8}")
tgat_results = {}
for k in FRACTIONS:
    m = train_tgat(k)
    tgat_results[k] = m
    print(f"{int(k*100):>6}%{m['accuracy']:>10.3f}{m['macro_f1']:>10.3f}{m['auc']:>8.3f}")

TGAT on politifact (early-stopped on val)
 window  accuracy  macro_f1     auc
    10%     0.498     0.362   0.809
    20%     0.489     0.328   0.608
    30%     0.507     0.373   0.812
    50%     0.489     0.328   0.522


SECTION 5.4 — Multi-seed TGAT (honest result: mean ± std)

In [ ]:
# One seed on 62 training graphs is a single draw from a high-variance
# distribution, not a result. We run several seeds and report mean ± std.
# Principled small-data adjustments: lr 5e-3 (was 1e-2); min-epochs before
# early stopping can fire.
SEEDS = [0, 1, 2, 3, 4]
LR = 5e-3

def run_tgat_cached(tensors, seed, lr=LR, epochs=300, patience=40, min_epochs=40):
    (Xtr,eitr,ettr,btr,ytr,ntr,
     Xva,eiva,etva,bva,yva,nva,
     Xte,eite,ette,bte,yte,nte) = tensors
    torch.manual_seed(seed)
    model = TGAT(in_dim=5).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, wait = -1, None, 0
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xtr,eitr,ettr,btr,ntr), ytr)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            va = torch.sigmoid(model(Xva,eiva,etva,bva,nva)).cpu().numpy()
        try: vauc = roc_auc_score(yva.cpu().numpy(), va)
        except: vauc = 0.5
        if vauc > best:
            best, best_state, wait = vauc, {kk:v.clone() for kk,v in model.state_dict().items()}, 0
        elif ep >= min_epochs:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        p = torch.sigmoid(model(Xte,eite,ette,bte,nte)).cpu().numpy()
    yhat=(p>=0.5).astype(int); yt=yte.cpu().numpy().astype(int)
    return (accuracy_score(yt,yhat), f1_score(yt,yhat,average='macro'), roc_auc_score(yt,p))

print(f"Multi-seed TGAT on {SUBSET} ({len(SEEDS)} seeds, lr={LR})")
print(f"{'window':>7}{'accuracy':>15}{'macro_f1':>15}{'auc':>15}")
tgat_ms = {}
for k in FRACTIONS:
    tens = (*graphs_to_tensors(train_idx,k), *graphs_to_tensors(val_idx,k),
            *graphs_to_tensors(test_idx,k))
    res = np.array([run_tgat_cached(tens, s) for s in SEEDS])   # (seeds, 3)
    mu, sd = res.mean(0), res.std(0)
    tgat_ms[k] = (mu.tolist(), sd.tolist())
    print(f"{int(k*100):>6}%   {mu[0]:.3f}±{sd[0]:.3f}  {mu[1]:.3f}±{sd[1]:.3f}  {mu[2]:.3f}±{sd[2]:.3f}")

Multi-seed TGAT on politifact (5 seeds, lr=0.005)
 window       accuracy       macro_f1            auc
    10%   0.611±0.123  0.532±0.188  0.811±0.030
    20%   0.628±0.097  0.573±0.145  0.811±0.017
    30%   0.714±0.099  0.688±0.137  0.847±0.034
    50%   0.658±0.118  0.606±0.179  0.755±0.063


SECTION 5.5 — Static GCN baseline (graph structure, NO time)

In [ ]:
# def run_gcn_cached(tensors, seed, lr=LR, epochs=300, patience=40, min_epochs=40):
#     (Xtr,eitr,ettr,btr,ytr,ntr, Xva,eiva,etva,bva,yva,nva, Xte,eite,ette,bte,yte,nte) = tensors
#     torch.manual_seed(seed)
#     model = FairGCN(in_dim=5).to(device)
#     opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
#     best, best_state, wait = -1, None, 0
#     for ep in range(epochs):
#         model.train(); opt.zero_grad()
#         loss = F.binary_cross_entropy_with_logits(model(Xtr,eitr,btr,ntr), ytr)  # no edge time
#         loss.backward(); opt.step()
#         model.eval()
#         with torch.no_grad():
#             va = torch.sigmoid(model(Xva,eiva,bva,nva)).cpu().numpy()
#         try: vauc = roc_auc_score(yva.cpu().numpy(), va)
#         except: vauc = 0.5
#         if vauc > best: best, best_state, wait = vauc, {k:v.clone() for k,v in model.state_dict().items()}, 0
#         elif ep >= min_epochs:
#             wait += 1
#             if wait >= patience: break
#     model.load_state_dict(best_state); model.eval()
#     with torch.no_grad(): p = torch.sigmoid(model(Xte,eite,bte,nte)).cpu().numpy()
#     yhat=(p>=0.5).astype(int); yt=yte.cpu().numpy().astype(int)
#     return (accuracy_score(yt,yhat), f1_score(yt,yhat,average='macro'), roc_auc_score(yt,p))

# print(f"Multi-seed FINAL fair GCN on {SUBSET} ({len(SEEDS)} seeds)")
# print(f"{'window':>7}{'accuracy':>15}{'macro_f1':>15}{'auc':>15}")
# gcn_ms = {}
# for k in FRACTIONS:
#     tens = (*graphs_to_tensors(train_idx,k), *graphs_to_tensors(val_idx,k), *graphs_to_tensors(test_idx,k))
#     res = np.array([run_gcn_cached(tens, s) for s in SEEDS])
#     mu, sd = res.mean(0), res.std(0)
#     gcn_ms[k] = (mu.tolist(), sd.tolist())
#     print(f"{int(k*100):>6}%   {mu[0]:.3f}±{sd[0]:.3f}  {mu[1]:.3f}±{sd[1]:.3f}  {mu[2]:.3f}±{sd[2]:.3f}")

# ===== SECTION 5.5 — Fair static GCN: DEFINITION + RUN in one cell =====
import warnings; warnings.filterwarnings("ignore")

def gcn_propagate(h, edge_index, num_nodes):
    s,d = edge_index[0], edge_index[1]; sl = torch.arange(num_nodes, device=h.device)
    s2 = torch.cat([s,d,sl]); d2 = torch.cat([d,s,sl])
    deg = torch.zeros(num_nodes, device=h.device).index_add_(0,d2,torch.ones(d2.size(0),device=h.device))
    di = deg.pow(-0.5); di[torch.isinf(di)]=0; norm = di[s2]*di[d2]
    return torch.zeros_like(h).index_add_(0,d2,norm.unsqueeze(-1)*h[s2])

def seg_mean(h,b,ng):
    H=h.size(1); m=torch.zeros(ng,H,device=h.device); c=torch.zeros(ng,1,device=h.device)
    m.index_add_(0,b,h); c.index_add_(0,b,torch.ones(b.size(0),1,device=h.device)); return m/(c+1e-9)
def seg_max(h,b,ng): return torch.full((ng,h.size(1)),-1e9,device=h.device).index_reduce_(0,b,h,'amax',include_self=True)
def seg_min(h,b,ng): return torch.full((ng,h.size(1)), 1e9,device=h.device).index_reduce_(0,b,h,'amin',include_self=True)
def flat_vector(x,b,ng):
    cnt=torch.zeros(ng,1,device=x.device).index_add_(0,b,torch.ones(b.size(0),1,device=x.device))
    return torch.cat([seg_mean(x,b,ng),seg_max(x,b,ng),seg_min(x,b,ng),cnt],dim=1)

class FairGCN(nn.Module):
    def __init__(self,in_dim,hid=32,layers=2,dropout=0.3):
        super().__init__()
        self.lins=nn.ModuleList([nn.Linear(in_dim if i==0 else hid,hid) for i in range(layers)])
        self.drop=nn.Dropout(dropout)
        self.head=nn.Sequential(nn.Linear(2*hid+3*in_dim+1,hid),nn.ReLU(),
                                nn.Dropout(dropout),nn.Linear(hid,1))
    def forward(self,x,edge_index,batch,ng):
        h=x
        for lin in self.lins:
            h=self.drop(F.relu(lin(gcn_propagate(h,edge_index,x.size(0)))))
        pooled=torch.cat([seg_mean(h,batch,ng),seg_max(h,batch,ng),flat_vector(x,batch,ng)],dim=1)
        return self.head(pooled).squeeze(-1)

def run_gcn_cached(tensors, seed, lr=LR, epochs=300, patience=40, min_epochs=40):
    (Xtr,eitr,ettr,btr,ytr,ntr, Xva,eiva,etva,bva,yva,nva, Xte,eite,ette,bte,yte,nte) = tensors
    torch.manual_seed(seed)
    model = FairGCN(in_dim=5).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, wait = -1, None, 0
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(Xtr,eitr,btr,ntr), ytr)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            va = torch.sigmoid(model(Xva,eiva,bva,nva)).cpu().numpy()
        try: vauc = roc_auc_score(yva.cpu().numpy(), va)
        except: vauc = 0.5
        if vauc > best: best, best_state, wait = vauc, {k:v.clone() for k,v in model.state_dict().items()}, 0
        elif ep >= min_epochs:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad(): p = torch.sigmoid(model(Xte,eite,bte,nte)).cpu().numpy()
    yhat=(p>=0.5).astype(int); yt=yte.cpu().numpy().astype(int)
    return (accuracy_score(yt,yhat), f1_score(yt,yhat,average='macro'), roc_auc_score(yt,p))

print(f"Multi-seed FINAL fair GCN on {SUBSET} ({len(SEEDS)} seeds)")
print(f"{'window':>7}{'accuracy':>15}{'macro_f1':>15}{'auc':>15}")
gcn_ms = {}
for k in FRACTIONS:
    tens = (*graphs_to_tensors(train_idx,k), *graphs_to_tensors(val_idx,k), *graphs_to_tensors(test_idx,k))
    res = np.array([run_gcn_cached(tens, s) for s in SEEDS])
    mu, sd = res.mean(0), res.std(0)
    gcn_ms[k] = (mu.tolist(), sd.tolist())
    print(f"{int(k*100):>6}%   {mu[0]:.3f}±{sd[0]:.3f}  {mu[1]:.3f}±{sd[1]:.3f}  {mu[2]:.3f}±{sd[2]:.3f}")

Multi-seed FINAL fair GCN on politifact (5 seeds)
 window       accuracy       macro_f1            auc
    10%   0.548±0.020  0.495±0.079  0.559±0.025
    20%   0.534±0.040  0.479±0.106  0.552±0.030
    30%   0.544±0.033  0.476±0.088  0.602±0.015
    50%   0.544±0.044  0.469±0.104  0.601±0.017


SECTION 5.6 — Ablation: does temporal information carry the signal?

In [ ]:
# The clean isolation (Objective 4 / RQ3): SAME TGAT architecture, once WITH
# temporal info and once with ALL of it removed (edge times zeroed + the two
# time node-features zeroed). Only time changes, so any gap is attributable to
# temporal information — not to architecture.

def ablate_split(split6):
    X, ei, et, b, y, n = split6
    Xa = X.clone(); Xa[:, 2] = 0.0; Xa[:, 3] = 0.0   # zero t_since_root, t_since_parent
    eta = torch.zeros_like(et)                        # zero edge times
    return Xa, ei, eta, b, y, n

def ablate_tensors(tensors):
    tr = ablate_split(tensors[0:6]); va = ablate_split(tensors[6:12]); te = ablate_split(tensors[12:18])
    return (*tr, *va, *te)

print(f"Ablation on {SUBSET}: TGAT WITH time vs TGAT NO-time ({len(SEEDS)} seeds)")
print(f"{'window':>7}{'AUC (with time)':>18}{'AUC (no time)':>16}{'gap':>8}")
ablation = {}
for k in FRACTIONS:
    tens = (*graphs_to_tensors(train_idx,k), *graphs_to_tensors(val_idx,k), *graphs_to_tensors(test_idx,k))
    tens_abl = ablate_tensors(tens)
    full = np.array([run_tgat_cached(tens, s)      for s in SEEDS])[:, 2]   # AUC col
    noti = np.array([run_tgat_cached(tens_abl, s)  for s in SEEDS])[:, 2]
    ablation[k] = (full.mean(), full.std(), noti.mean(), noti.std())
    print(f"{int(k*100):>6}%   {full.mean():.3f}±{full.std():.3f}    "
          f"{noti.mean():.3f}±{noti.std():.3f}   {full.mean()-noti.mean():+.3f}")

Ablation on politifact: TGAT WITH time vs TGAT NO-time (5 seeds)
 window   AUC (with time)   AUC (no time)     gap
    10%   0.815±0.032    0.560±0.026   +0.255
    20%   0.792±0.038    0.589±0.011   +0.203
    30%   0.864±0.038    0.549±0.021   +0.315
    50%   0.804±0.020    0.513±0.013   +0.292


SECTION 5.6b — Ablation DECOMPOSITION (speed vs depth vs breadth)

In [ ]:
# ===== SECTION 5.6b — Ablation DECOMPOSITION (speed vs depth vs breadth) =====
# Objective 4 / RQ3: WHICH diffusion property carries the detection signal?
# Feature columns: [0]=depth [1]=degree [2]=t_since_root [3]=t_since_parent [4]=subtree
#   SPEED   = timing: cols 2,3 + edge times      DEPTH = col 0
#   BREADTH = cols 1,4 (degree, subtree)
# Leave-one-out: full vs remove-each-property. Bigger drop = more important.

# --- reload PolitiFact explicitly (decomposition is informative where the
#     temporal effect is large; GossipCop is saved on disk, safe to overwrite) ---
SUBSET = "politifact"
node_graph_id, graph_labels, node_time, edges = load_subset(SUBSET)
G = graph_labels.shape[0]
order = np.argsort(node_graph_id, kind="stable"); sg = node_graph_id[order]
bnd = np.searchsorted(sg, np.arange(G+1))
nodes_of = {g: order[bnd[g]:bnd[g+1]] for g in range(G)}
edge_gid = node_graph_id[edges[:,0]]; edges_of = defaultdict(list)
for (u,v),g in zip(edges, edge_gid): edges_of[g].append((int(u),int(v)))
train_idx, val_idx, test_idx = load_split(SUBSET)
print(f"reloaded {SUBSET}: train {len(train_idx)}, val {len(val_idx)}, test {len(test_idx)}")
print(f"using SEEDS = {SEEDS}")   # confirm 5 seeds for the final numbers

def ablate_split_cols(split6, zero_cols, zero_time):
    X, ei, et, b, y, n = split6
    Xa = X.clone()
    for c in zero_cols: Xa[:, c] = 0.0
    eta = torch.zeros_like(et) if zero_time else et
    return Xa, ei, eta, b, y, n

def ablate_cols(tensors, zero_cols, zero_time):
    tr = ablate_split_cols(tensors[0:6],  zero_cols, zero_time)
    va = ablate_split_cols(tensors[6:12], zero_cols, zero_time)
    te = ablate_split_cols(tensors[12:18], zero_cols, zero_time)
    return (*tr, *va, *te)

VARIANTS = {
    "full":       ([],    False),
    "no_speed":   ([2,3], True),
    "no_depth":   ([0],   False),
    "no_breadth": ([1,4], False),
}

print(f"\nAblation decomposition on {SUBSET} ({len(SEEDS)} seeds) - test AUC")
print(f"{'window':>7}" + "".join(f"{v:>15}" for v in VARIANTS))
decomp = {}
for k in FRACTIONS:
    tens = (*graphs_to_tensors(train_idx,k), *graphs_to_tensors(val_idx,k), *graphs_to_tensors(test_idx,k))
    row = {}
    for name,(cols,zt) in VARIANTS.items():
        tv = ablate_cols(tens, cols, zt) if (cols or zt) else tens
        aucs = np.array([run_tgat_cached(tv, s) for s in SEEDS])[:,2]
        row[name] = (float(aucs.mean()), float(aucs.std()))
    decomp[k] = row
    print(f"{int(k*100):>6}%" + "".join(f"  {row[v][0]:.3f}±{row[v][1]:.3f}" for v in VARIANTS))

print("\nMean AUC DROP from full, averaged over windows (bigger = more important):")
for name in ["no_speed","no_depth","no_breadth"]:
    drop = np.mean([decomp[k]["full"][0] - decomp[k][name][0] for k in FRACTIONS])
    print(f"  removing {name.replace('no_',''):>8}: {drop:+.3f}")

reloaded politifact: train 62, val 31, test 221
using SEEDS = [0, 1, 2, 3, 4]

Ablation decomposition on politifact (5 seeds) - test AUC
 window           full       no_speed       no_depth     no_breadth
    10%  0.768±0.062  0.564±0.028  0.811±0.023  0.849±0.031
    20%  0.794±0.039  0.586±0.006  0.805±0.038  0.847±0.026
    30%  0.855±0.028  0.565±0.019  0.824±0.036  0.821±0.082
    50%  0.720±0.090  0.513±0.013  0.807±0.091  0.806±0.080

Mean AUC DROP from full, averaged over windows (bigger = more important):
  removing    speed: +0.227
  removing    depth: -0.027
  removing  breadth: -0.046


SECTION 5.7 — LSTM baseline (temporal, NO graph)

In [ ]:
# The missing corner of the 2x2: temporal ordering WITHOUT graph structure.
# Reads each cascade as a chronological SEQUENCE of node features. Comparing
# TGAT vs LSTM isolates the value of graph structure on top of temporal order.
class SeqLSTM(nn.Module):
    def __init__(self, in_dim, hid=32, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hid, batch_first=True)
        self.head = nn.Sequential(nn.Linear(hid, hid), nn.ReLU(),
                                  nn.Dropout(dropout), nn.Linear(hid, 1))
    def forward(self, padded, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(padded, lengths.cpu(),
                     batch_first=True, enforce_sorted=False)
        _, (hn, _) = self.lstm(packed)
        return self.head(hn[-1]).squeeze(-1)

# build padded sequences for a set of graphs at window k (masking-aware norm per graph)
def seq_dataset(graph_ids, k):
    seqs, lengths, ys = [], [], []
    for g in graph_ids:
        r = build_representations(int(g), k)
        seq = r["sequence"]                        # chronological node-feature rows
        if len(seq) == 0:
            seq = [tuple(r["static"]["feats"][0])]  # guard: at least the root
        arr = normalise_observed([list(f) for f in seq])
        seqs.append(torch.tensor(arr, dtype=torch.float))
        lengths.append(len(seq)); ys.append(r["label"])
    padded = nn.utils.rnn.pad_sequence(seqs, batch_first=True).to(device)
    return padded, torch.tensor(lengths), torch.tensor(np.array(ys), dtype=torch.float, device=device)

def run_lstm(k, seed, lr=LR, epochs=300, patience=40, min_epochs=40):
    torch.manual_seed(seed)
    ptr,ltr,ytr = seq_dataset(train_idx, k)
    pva,lva,yva = seq_dataset(val_idx, k)
    pte,lte,yte = seq_dataset(test_idx, k)
    model = SeqLSTM(5).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, wait = -1, None, 0
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(ptr,ltr), ytr)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            va = torch.sigmoid(model(pva,lva)).cpu().numpy()
        try: vauc = roc_auc_score(yva.cpu().numpy(), va)
        except: vauc = 0.5
        if vauc > best: best, best_state, wait = vauc, {kk:v.clone() for kk,v in model.state_dict().items()}, 0
        elif ep >= min_epochs:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad(): p = torch.sigmoid(model(pte,lte)).cpu().numpy()
    yhat=(p>=0.5).astype(int); yt=yte.cpu().numpy().astype(int)
    return (accuracy_score(yt,yhat), f1_score(yt,yhat,average='macro'), roc_auc_score(yt,p))

print(f"Multi-seed LSTM (sequence, no graph) on {SUBSET} ({len(SEEDS)} seeds)")
print(f"{'window':>7}{'accuracy':>15}{'macro_f1':>15}{'auc':>15}")
lstm_ms = {}
for k in FRACTIONS:
    res = np.array([run_lstm(k, s) for s in SEEDS])
    mu, sd = res.mean(0), res.std(0)
    lstm_ms[k] = (mu.tolist(), sd.tolist())
    print(f"{int(k*100):>6}%   {mu[0]:.3f}±{sd[0]:.3f}  {mu[1]:.3f}±{sd[1]:.3f}  {mu[2]:.3f}±{sd[2]:.3f}")

Multi-seed LSTM (sequence, no graph) on politifact (5 seeds)
 window       accuracy       macro_f1            auc
    10%   0.505±0.020  0.473±0.034  0.522±0.031
    20%   0.517±0.025  0.497±0.051  0.503±0.023
    30%   0.547±0.009  0.541±0.015  0.552±0.021
    50%   0.544±0.033  0.530±0.025  0.558±0.034


SECTION 5.8 — Save results to Drive (JSON)

In [ ]:
import json, os
from datetime import datetime

def to_serialisable(obj):
    # handles dicts of tuples/arrays/np types -> plain python
    import numpy as np
    if isinstance(obj, dict):  return {str(k): to_serialisable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [to_serialisable(v) for v in obj]
    if isinstance(obj, (np.floating, np.integer)): return float(obj)
    return obj

results = {}
for name in ["lr_results", "tgat_ms", "gcn_ms", "ablation", "lstm_ms"]:
    if name in globals():
        results[name] = to_serialisable(globals()[name])
        print(f"  found and saving: {name}")
    else:
        print(f"  (not in memory, skipping: {name})")

results["_meta"] = {"subset": SUBSET, "seeds": SEEDS, "fractions": FRACTIONS,
                    "saved_at": datetime.now().isoformat(),
                    "device": str(device)}

out_path = os.path.join(SAVE_DIR, f"results_{SUBSET}.json")
with open(out_path, "w") as f:

    json.dump(results, f, indent=2)
print("\nSaved to:", out_path)

  found and saving: lr_results
  found and saving: tgat_ms
  found and saving: gcn_ms
  found and saving: ablation
  found and saving: lstm_ms

Saved to: /content/drive/MyDrive/thesis_tagfn/results_politifact.json


**SECTION 6.0 — Scalability study, **
Stage 1 (TGAT cost vs dataset scale)

In [ ]:
# ===== SECTION 6.0 — Scalability study, Stage 1 (TGAT cost vs dataset scale) =====
# Measures preprocessing time, training time, and PEAK GPU MEMORY as dataset
# scale grows, using increasing samples of Fakeddit. Output = a scaling curve
# (RQ4). Fakeddit is the SCALABILITY tier only (tiny cascades -> no early-detection).
import time, gc

# --- load Fakeddit structural arrays (small files; not the text parquet) ---
fk_ngi   = np.load(hf_hub_download(REPO, "fakeddit/node_graph_id.npy", repo_type="dataset"))
fk_lab   = np.load(hf_hub_download(REPO, "fakeddit/graph_labels.npy",  repo_type="dataset")).astype(int).ravel()
fk_time  = np.load(hf_hub_download(REPO, "fakeddit/node_time.npy",     repo_type="dataset"))
fk_apath = hf_hub_download(REPO, "fakeddit/A.txt", repo_type="dataset")
fk_edges = []
with open(fk_apath) as fh:
    for line in fh:
        line = line.strip()
        if not line: continue
        if "," in line: x, y = line.split(",")[:2]
        else:           p = line.split(); x, y = p[0], p[1]
        fk_edges.append((int(x), int(y)))
fk_edges = np.array(fk_edges, dtype=np.int64)
FK_G = fk_lab.shape[0]
print(f"Fakeddit loaded: {FK_G:,} graphs, {fk_ngi.shape[0]:,} nodes, {fk_edges.shape[0]:,} edges")

# --- group once (vectorised) ---
fk_order = np.argsort(fk_ngi, kind="stable")
fk_sorted = fk_ngi[fk_order]
fk_bnd = np.searchsorted(fk_sorted, np.arange(FK_G + 1))
fk_nodes_of = {g: fk_order[fk_bnd[g]:fk_bnd[g+1]] for g in range(FK_G)}
fk_egid = fk_ngi[fk_edges[:, 0]]
fk_edges_of = defaultdict(list)
for (u, v), g in zip(fk_edges, fk_egid):
    fk_edges_of[g].append((int(u), int(v)))
print("grouped.")

# --- build TGAT tensors for a sample of graphs (full observation, since this is
#     a throughput test, not early detection) ---
def fk_build_tensors(graph_ids):
    xs, srcs, dsts, ets, batch, ys = [], [], [], [], [], []
    off = 0; gi = 0
    for g in graph_ids:
        root = int(fk_nodes_of[g].min())
        tmap = {int(n): int(fk_time[n]) for n in fk_nodes_of[g]}
        node_list = sorted(tmap.keys()); idx = {n:i for i,n in enumerate(node_list)}
        # minimal structural+temporal features via a quick BFS
        adj = defaultdict(list)
        for (u,v) in fk_edges_of[g]:
            adj[u].append(v); adj[v].append(u)
        parent={root:None}; depth={root:0}; bfs=[]; q=deque([root]); seen={root}
        while q:
            u=q.popleft(); bfs.append(u)
            for w in adj[u]:
                if w not in seen: seen.add(w); parent[w]=u; depth[w]=depth[u]+1; q.append(w)
        sub={n:1 for n in node_list}
        for u in reversed(bfs):
            if parent.get(u) is not None: sub[parent[u]]+=sub[u]
        nonroot=[tmap[n] for n in node_list if n!=root]
        t0=min(nonroot) if nonroot else 0
        rt=lambda n:0 if n==root else tmap[n]-t0
        feats={n:[depth.get(n,-1),len(adj[n]),rt(n),
                  0 if parent.get(n) is None else rt(n)-rt(parent[n]),sub[n]] for n in node_list}
        xs.append(normalise_observed([feats[n] for n in node_list]))
        for (u,v) in fk_edges_of[g]:
            c,pnode = (u,v) if tmap.get(u,0)>=tmap.get(v,0) else (v,u)
            srcs.append(idx[c]+off); dsts.append(idx[pnode]+off); ets.append(tmap.get(c,0))
        batch += [gi]*len(node_list); ys.append(int(fk_lab[g])); off+=len(node_list); gi+=1
    X  = torch.tensor(np.concatenate(xs), dtype=torch.float, device=device)
    ei = torch.tensor(np.array([srcs,dsts]), dtype=torch.long, device=device)
    et = torch.tensor(np.log1p(np.array(ets,dtype=float)), dtype=torch.float, device=device).unsqueeze(1)
    b  = torch.tensor(np.array(batch), dtype=torch.long, device=device)
    y  = torch.tensor(np.array(ys), dtype=torch.float, device=device)
    return X, ei, et, b, y, gi

SCALE_SIZES = [1000, 5000, 10000, 25000, 50000]
EPOCHS_TIMED = 20            # a fixed number of epochs, for a clean time-per-epoch measure

print(f"\n{'n_graphs':>9}{'prep_s':>9}{'train_s':>9}{'s/epoch':>9}{'peak_MB':>10}{'graphs/s':>10}")
scalability = {}
rng = np.random.default_rng(0)
for n in SCALE_SIZES:
    sample = rng.choice(FK_G, size=n, replace=False)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    t0 = time.perf_counter()
    X, ei, et, b, y, ng = fk_build_tensors(sample)
    prep = time.perf_counter() - t0
    torch.manual_seed(0)
    model = TGAT(in_dim=5).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=1e-4)
    t1 = time.perf_counter()
    for ep in range(EPOCHS_TIMED):
        model.train(); opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model(X,ei,et,b,ng), y)
        loss.backward(); opt.step()
    if device.type == "cuda": torch.cuda.synchronize()
    train = time.perf_counter() - t1
    peak_mb = (torch.cuda.max_memory_allocated()/1e6) if device.type=="cuda" else float("nan")
    scalability[n] = {"prep_s":prep,"train_s":train,"s_per_epoch":train/EPOCHS_TIMED,
                      "peak_mb":peak_mb,"graphs_per_s":n/(train/EPOCHS_TIMED)}
    print(f"{n:>9,}{prep:>9.2f}{train:>9.2f}{train/EPOCHS_TIMED:>9.3f}{peak_mb:>10.0f}{n/(train/EPOCHS_TIMED):>10,.0f}")
    del X, ei, et, b, y, model, opt; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()

fakeddit/node_graph_id.npy: reconstructing file:   0%|          |  0.00B / 58.0MB            

fakeddit/node_graph_id.npy: downloading bytes:           |  0.00B            

fakeddit/graph_labels.npy: reconstructing file:   0%|          |  0.00B / 4.53MB            

fakeddit/graph_labels.npy: downloading bytes:           |  0.00B            

fakeddit/node_time.npy: reconstructing file:   0%|          |  0.00B / 58.0MB            

fakeddit/node_time.npy: downloading bytes:           |  0.00B            

fakeddit/A.txt: reconstructing file:   0%|          |  0.00B /  112MB            

fakeddit/A.txt: downloading bytes:           |  0.00B            

Fakeddit loaded: 566,104 graphs, 7,249,803 nodes, 6,683,699 edges
grouped.

 n_graphs   prep_s  train_s  s/epoch   peak_MB  graphs/s
    1,000     0.14     0.15    0.008        75   132,945
    5,000     0.86     0.35    0.017       275   289,217
   10,000     2.28     0.66    0.033       513   304,152
   25,000     3.82     1.59    0.079      1259   315,081
   50,000     9.23     3.26    0.163      2525   307,018


SECTION 6.1 — Extend the scalability curve + find the T4 ceiling

In [ ]:
# ===== SECTION 6.1 — Extend the scalability curve + find the T4 ceiling =====
import time, gc

EXTRA_SIZES = [100000, 200000, 300000]
EPOCHS_TIMED = 20
rng = np.random.default_rng(1)

print(f"{'n_graphs':>9}{'prep_s':>9}{'s/epoch':>9}{'peak_MB':>10}{'graphs/s':>11}   status")
for n in EXTRA_SIZES:
    if n > FK_G:
        print(f"{n:>9,}  (exceeds Fakeddit size {FK_G:,}, skipped)"); continue
    try:
        sample = rng.choice(FK_G, size=n, replace=False)
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
        t0 = time.perf_counter()
        X, ei, et, b, y, ng = fk_build_tensors(sample)
        prep = time.perf_counter() - t0
        torch.manual_seed(0)
        model = TGAT(in_dim=5).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=5e-3, weight_decay=1e-4)
        t1 = time.perf_counter()
        for ep in range(EPOCHS_TIMED):
            model.train(); opt.zero_grad()
            loss = F.binary_cross_entropy_with_logits(model(X,ei,et,b,ng), y)
            loss.backward(); opt.step()
        if device.type == "cuda": torch.cuda.synchronize()
        train = time.perf_counter() - t1
        peak_mb = (torch.cuda.max_memory_allocated()/1e6) if device.type=="cuda" else float("nan")
        scalability[n] = {"prep_s":prep,"train_s":train,"s_per_epoch":train/EPOCHS_TIMED,
                          "peak_mb":peak_mb,"graphs_per_s":n/(train/EPOCHS_TIMED)}
        print(f"{n:>9,}{prep:>9.2f}{train/EPOCHS_TIMED:>9.3f}{peak_mb:>10.0f}"
              f"{n/(train/EPOCHS_TIMED):>11,.0f}   OK")
        del X, ei, et, b, y, model, opt; gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"{n:>9,}   --- OUT OF MEMORY (this is the T4 full-batch ceiling) ---")
            scalability[f"{n}_OOM"] = True
            if device.type == "cuda": torch.cuda.empty_cache(); gc.collect()
            break
        else:
            raise

# save the whole scalability result to Drive
import json, os
with open(os.path.join(SAVE_DIR, "scalability_fakeddit.json"), "w") as f:
    json.dump({str(k): v for k, v in scalability.items()}, f, indent=2)
print("\nsaved -> scalability_fakeddit.json")

 n_graphs   prep_s  s/epoch   peak_MB   graphs/s   status
  100,000    15.58    0.265      4974    377,693   OK
  200,000    31.16    0.530     10013    377,464   OK
  300,000   --- OUT OF MEMORY (this is the T4 full-batch ceiling) ---

saved -> scalability_fakeddit.json


DeLong (the rigorous per-cascade AUC comparison) and a paired t-test across your 5 seeds

In [ ]:
# ===== SIGNIFICANCE TESTS (PolitiFact, CPU-fine) =====
# Reuses your TGAT, FairGCN, graphs_to_tensors, train/val/test idx, SEEDS.
import numpy as np, torch, torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from scipy import stats

# --- DeLong test for two correlated AUCs (verified) ---
def _midrank(x):
    J=np.argsort(x); Z=x[J]; N=len(x); T=np.zeros(N); i=0
    while i<N:
        j=i
        while j<N and Z[j]==Z[i]: j+=1
        T[i:j]=0.5*(i+j-1)+1; i=j
    T2=np.empty(N); T2[J]=T; return T2
def delong(y,s1,s2):
    y=np.asarray(y); pos=y==1; neg=y==0; m=pos.sum(); n=neg.sum()
    P=np.vstack((s1,s2)); ps=P[:,pos]; ns=P[:,neg]
    tx=np.array([_midrank(ps[r]) for r in range(2)])
    ty=np.array([_midrank(ns[r]) for r in range(2)])
    tz=np.array([_midrank(np.concatenate((ps[r],ns[r]))) for r in range(2)])
    aucs=tz[:,:m].sum(1)/(m*n)-(m+1.0)/(2.0*n)
    v01=(tz[:,:m]-tx)/n; v10=1.0-(tz[:,m:]-ty)/m
    s=np.cov(v01)/m+np.cov(v10)/n
    var=s[0,0]+s[1,1]-2*s[0,1]
    z=(aucs[0]-aucs[1])/np.sqrt(var+1e-12)
    return aucs[0],aucs[1],2*(1-stats.norm.cdf(abs(z)))

def _fit_predict(kind, k, seed, ablate=False, epochs=300, patience=40, min_epochs=40, lr=5e-3):
    torch.manual_seed(seed)
    Xtr,eitr,ettr,btr,ytr,ntr = graphs_to_tensors(train_idx,k)
    Xva,eiva,etva,bva,yva,nva = graphs_to_tensors(val_idx,k)
    Xte,eite,ette,bte,yte,nte = graphs_to_tensors(test_idx,k)
    if ablate:
        for T in (Xtr,Xva,Xte): T[:,2]=0.0; T[:,3]=0.0
        ettr=torch.zeros_like(ettr); etva=torch.zeros_like(etva); ette=torch.zeros_like(ette)
    model = (TGAT(in_dim=5) if kind!="gcn" else FairGCN(in_dim=5)).to(device)
    opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=1e-4)
    def fwd(X,ei,et,b,ng): return model(X,ei,et,b,ng) if kind!="gcn" else model(X,ei,b,ng)
    best,bs,wait=-1,None,0
    for ep in range(epochs):
        model.train(); opt.zero_grad()
        loss=F.binary_cross_entropy_with_logits(fwd(Xtr,eitr,ettr,btr,ntr),ytr)
        loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): va=torch.sigmoid(fwd(Xva,eiva,etva,bva,nva)).cpu().numpy()
        try: vauc=roc_auc_score(yva.cpu().numpy(),va)
        except: vauc=0.5
        if vauc>best: best,bs,wait=vauc,{k2:v.clone() for k2,v in model.state_dict().items()},0
        elif ep>=min_epochs:
            wait+=1
            if wait>=patience: break
    model.load_state_dict(bs); model.eval()
    with torch.no_grad(): p=torch.sigmoid(fwd(Xte,eite,ette,bte,nte)).cpu().numpy()
    return p, yte.cpu().numpy()

K = 0.10   # earliest window — where early detection matters most
print(f"Significance tests on PolitiFact @ {int(K*100)}% window\n")

# per-cascade probabilities from a representative seed (seed 0) for DeLong
p_tgat, y = _fit_predict("tgat", K, 0)
p_gcn,  _ = _fit_predict("gcn",  K, 0)
p_notime,_= _fit_predict("tgat", K, 0, ablate=True)

for name, pa, pb in [("TGAT vs static GCN", p_tgat, p_gcn),
                     ("TGAT with-time vs no-time", p_tgat, p_notime)]:
    a1,a2,pv = delong(y, pa, pb)
    print(f"DeLong  {name}: AUC {a1:.3f} vs {a2:.3f}, p = {pv:.2e}  {'(significant)' if pv<0.05 else '(NOT sig)'}")

# paired t-test across the 5 seeds
def seed_aucs(kind, ablate=False):
    out=[]
    for s in SEEDS:
        p,yy=_fit_predict(kind,K,s,ablate=ablate); out.append(roc_auc_score(yy,p))
    return np.array(out)
a_tgat=seed_aucs("tgat"); a_gcn=seed_aucs("gcn"); a_nt=seed_aucs("tgat",ablate=True)
t1,pp1=stats.ttest_rel(a_tgat,a_gcn); t2,pp2=stats.ttest_rel(a_tgat,a_nt)
print(f"\nPaired t-test (5 seeds)  TGAT vs GCN:      t={t1:.2f}, p={pp1:.2e}")
print(f"Paired t-test (5 seeds)  TGAT vs no-time:  t={t2:.2f}, p={pp2:.2e}")

Significance tests on PolitiFact @ 10% window

DeLong  TGAT vs static GCN: AUC 0.847 vs 0.589, p = 1.44e-08  (significant)
DeLong  TGAT with-time vs no-time: AUC 0.847 vs 0.592, p = 4.56e-08  (significant)

Paired t-test (5 seeds)  TGAT vs GCN:      t=9.97, p=5.68e-04
Paired t-test (5 seeds)  TGAT vs no-time:  t=10.19, p=5.23e-04
